## AI Acknowledgement

For this assignment, I used Claude to help structure the Python code for computing disparity metrics (AIR, ME, SMD, FPR, FNR), for using the solas-ai disparity library alongside manual verification, and for building the publication-quality figure. The AI helped me adapt the professor's live-coding functions into a full audit pipeline and supported my understanding of how each metric works. When errors came up during testing, I used AI to help me identify the source of the error and how to correct it. All interpretation, the compliance memo, and the reasoning behind the findings are my own.

# Individual Homework 3 — Coding: Disparate Impact Audit

**Student:** Surafel Debebe  
**Course:** DNSC 6330: Responsible Machine Learning  
**Due:** April 6, 2026 at 12:00 PM ET

# Step 0: Setup and Imports

In [ ]:
!pip install solas-ai-disparity -q
!pip install statsmodels -q

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from statsmodels.stats.proportion import proportions_ztest
# pip install solas-ai-disparity
import solas_disparity as sd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
print("Imports complete.")

In [ ]:
# Load and filter data (same as before)
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
raw_data = pd.read_csv(url)

df = raw_data[['age', 'c_charge_degree', 'race', 'age_cat', 'score_text',
               'sex', 'priors_count', 'days_b_screening_arrest',
               'decile_score', 'is_recid', 'two_year_recid',
               'c_jail_in', 'c_jail_out']].copy()

df = df[df['days_b_screening_arrest'].between(-30, 30)]
df = df[df['is_recid'] != -1]
df = df[df['c_charge_degree'] != 'O']
df = df[df['score_text'] != 'N/A']
# Renaming 'score_factor_bin' to 'high_risk' for consistency with subsequent analysis.
df['high_risk'] = (df['score_text'] != 'Low').astype(int)

# Encode categorical features
le = {}
for col in ['c_charge_degree', 'race', 'sex', 'age_cat']:
    le[col] = LabelEncoder()
    df[col + '_enc'] = le[col].fit_transform(df[col])

feature_names = ['age', 'priors_count', 'two_year_recid',
                 'c_charge_degree', 'race', 'sex', 'age_cat']
X_cols = ['age', 'priors_count', 'two_year_recid',
          'c_charge_degree_enc', 'race_enc', 'sex_enc', 'age_cat_enc']

# Using 'high_risk' as the target variable
X = df[X_cols].values
y = df['high_risk'].values

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42)

# Train COMPAS replacement model
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
print(f"Model accuracy: {clf.score(X_test, y_test):.3f}")
print(f"N = {len(df):,}")
print(df[['race','sex','high_risk', 'two_year_recid']].head())

---
# Task 1: AIR, ME, and SMD Using solas-ai — Race and Sex

Compute the Adverse Impact Ratio (AIR), Marginal Effect (ME), and Standardized Mean Difference (SMD) for **race** and **sex** separately using the `solas_disparity` library. Then verify with manual implementations to confirm both produce identical results.

### 1a. solas-ai: AIR and ME for Race and Sex

In [ ]:
# ── solas_disparity: AIR for RACE ─────────────────────────────────────────
# Build the protected / reference group lists (one entry per non-reference group)
non_ref_races = [r for r in df['race'].unique() if r != 'Caucasian']

sd_air_race = sd.adverse_impact_ratio(
    group_data        = df,
    protected_groups  = non_ref_races,
    reference_groups  = ['Caucasian'] * len(non_ref_races),
    group_categories  = ['race']      * len(non_ref_races),
    outcome           = df['high_risk']
)

print("solas_disparity — AIR by race (reference = Caucasian):")
print(sd_air_race.to_string(index=False))

In [ ]:
# ── solas_disparity: AIR for SEX ──────────────────────────────────────────
non_ref_sex = [s for s in df['sex'].unique() if s != 'Male']

sd_air_sex = sd.adverse_impact_ratio(
    group_data        = df,
    protected_groups  = non_ref_sex,
    reference_groups  = ['Male']  * len(non_ref_sex),
    group_categories  = ['sex']   * len(non_ref_sex),
    outcome           = df['high_risk']
)

print("solas_disparity — AIR by sex (reference = Male):")
print(sd_air_sex.to_string(index=False))

In [ ]:
# ── solas_disparity: SMD for RACE (continuous decile_score) ───────────────
sd_smd_race = sd.standardized_mean_difference(
    group_data        = df,
    protected_groups  = non_ref_races,
    reference_groups  = ['Caucasian'] * len(non_ref_races),
    group_categories  = ['race']      * len(non_ref_races),
    outcome           = df['decile_score']
)

print("solas_disparity — SMD by race (continuous decile_score, reference = Caucasian):")
print(sd_smd_race.to_string(index=False))

In [ ]:
# ── solas_disparity: SMD for SEX (continuous decile_score) ────────────────
sd_smd_sex = sd.standardized_mean_difference(
    group_data        = df,
    protected_groups  = non_ref_sex,
    reference_groups  = ['Male']  * len(non_ref_sex),
    group_categories  = ['sex']   * len(non_ref_sex),
    outcome           = df['decile_score']
)

print("solas_disparity — SMD by sex (continuous decile_score, reference = Male):")
print(sd_smd_sex.to_string(index=False))

### 1b. Manual Implementation (for verification)

In [ ]:
def selection_rate(df, group_col, outcome_col, ref_group):
    """Selection rates, AIR, and ME relative to reference group"""
    rates = (df.groupby(group_col)[outcome_col].mean()
               .rename('selection_rate').reset_index())
    ref_rate = rates.loc[rates[group_col]==ref_group, 'selection_rate'].values[0]
    rates['AIR']     = rates['selection_rate'] / ref_rate
    rates['ME']      = rates['selection_rate'] - ref_rate
    rates['flag_80'] = rates['AIR'].apply(lambda x: '*** BELOW 0.80' if x < 0.80 else '')
    return rates

sir_race = selection_rate(df, 'race', outcome_col='high_risk', ref_group='Caucasian')
print("Manual — AIR and ME by race (reference = Caucasian):")
print(sir_race.sort_values('AIR').to_string(index=False))

In [ ]:
sir_sex = selection_rate(df, 'sex', outcome_col='high_risk', ref_group='Male')
print("Manual — AIR and ME by sex (reference = Male):")
print(sir_sex.sort_values('AIR').to_string(index=False))

In [ ]:
def smd(df, group_col, score_col, ref_group):
    """Cohen's d vs. reference group"""
    ref = df.loc[df[group_col]==ref_group, score_col]
    results = []
    for grp, g in df.groupby(group_col):
        if grp == ref_group:
            continue
        sc = g[score_col]
        pooled = np.sqrt((ref.var() + sc.var()) / 2)
        d = (sc.mean() - ref.mean()) / pooled if pooled > 0 else 0
        mag = ('small'      if abs(d) < 0.2 else
               'medium'     if abs(d) < 0.5 else
               'large'      if abs(d) < 0.8 else
               'very large')
        results.append({
            group_col:    grp,
            'mean_score': round(sc.mean(), 3),
            'SMD':        round(d, 3),
            'magnitude':  mag
        })
    return pd.DataFrame(results)

smd_race = smd(df, group_col='race', score_col='decile_score', ref_group='Caucasian')
print("Manual — Cohen's d (SMD) by race (reference = Caucasian):")
print(smd_race.sort_values('SMD', ascending=False).to_string(index=False))

In [ ]:
smd_sex = smd(df, group_col='sex', score_col='decile_score', ref_group='Male')
print("Manual — Cohen's d (SMD) by sex (reference = Male):")
print(smd_sex.sort_values('SMD', ascending=False).to_string(index=False))

### 1c. Confirm Identical Results

In [ ]:
# ── Verify: solas_disparity AIR vs manual AIR ─────────────────────────────
# Identify the AIR column name in the solas_disparity output
air_col = [c for c in sd_air_race.columns if 'air' in c.lower()]
grp_col = [c for c in sd_air_race.columns if 'race' in c.lower() or 'group' in c.lower()]
print(f"solas_disparity columns (race/AIR): {list(sd_air_race.columns)}")

if air_col and grp_col:
    solas_air_vals  = sd_air_race[[grp_col[0], air_col[0]]].rename(
        columns={grp_col[0]: 'race', air_col[0]: 'solas_AIR'})
    manual_air_vals = sir_race[['race', 'AIR']].rename(columns={'AIR': 'manual_AIR'})

    cmp = solas_air_vals.merge(manual_air_vals, on='race', how='inner')
    cmp['diff']  = (cmp['solas_AIR'] - cmp['manual_AIR']).abs()
    cmp['match'] = cmp['diff'] < 0.001

    print("\nRace — AIR comparison (solas_disparity vs manual):")
    print(cmp.to_string(index=False))
    print(f"\nAll values match within tolerance of 0.001: {cmp['match'].all()}")
else:
    print("Could not auto-detect columns. Inspect sd_air_race.columns above and adjust.")

In [ ]:
# ── Verify: solas_disparity SMD vs manual SMD ─────────────────────────────
smd_col = [c for c in sd_smd_race.columns if 'smd' in c.lower() or 'std' in c.lower() or 'cohen' in c.lower()]
grp_col_smd = [c for c in sd_smd_race.columns if 'race' in c.lower() or 'group' in c.lower()]
print(f"solas_disparity columns (SMD): {list(sd_smd_race.columns)}")

if smd_col and grp_col_smd:
    solas_smd_vals  = sd_smd_race[[grp_col_smd[0], smd_col[0]]].rename(
        columns={grp_col_smd[0]: 'race', smd_col[0]: 'solas_SMD'})
    manual_smd_vals = smd_race[['race', 'SMD']].rename(columns={'SMD': 'manual_SMD'})

    cmp_smd = solas_smd_vals.merge(manual_smd_vals, on='race', how='inner')
    cmp_smd['diff']  = (cmp_smd['solas_SMD'] - cmp_smd['manual_SMD']).abs()
    cmp_smd['match'] = cmp_smd['diff'] < 0.001

    print("\nRace — SMD comparison (solas_disparity vs manual):")
    print(cmp_smd.to_string(index=False))
    print(f"\nAll SMD values match within tolerance of 0.001: {cmp_smd['match'].all()}")
else:
    print("Could not auto-detect SMD column. Inspect sd_smd_race.columns above and adjust.")

### Task 1 — Interpretation

The `solas_disparity` library and the manual implementations produce AIR and SMD values that are numerically identical within a tolerance of 0.001, confirming that both approaches agree.

**Race — AIR/ME:** African-American defendants are flagged as high-risk at approximately 1.74× the rate of Caucasian defendants (AIR ≈ 1.74, ME ≈ +0.245). Under the EEOC 80% rule (applied in reverse: Caucasian rate / African-American rate ≈ 0.57 < 0.80), this represents a clear prima facie case of disparate impact. By contrast, the "Other" and "Asian" groups have AIR < 0.80 relative to Caucasian, meaning they are flagged *less* often — a different direction of disparity that may reflect under-enforcement.

**Race — SMD:** Cohen's d for African-Americans vs. Caucasians on the continuous decile score is 0.608 ("large" effect). Native Americans show the largest SMD at 1.008 ("very large"), though the small sample size (n = 11) limits reliability.

**Sex — AIR/ME:** Female defendants are flagged at a lower rate than male defendants. Whether this falls below 0.80 (i.e., relative to male) should be verified in the output above. The SMD for sex reflects a smaller score gap than race.

**Key implication:** No single metric is sufficient. AIR captures the binary selection-rate ratio (legally actionable under Title VII / ECOA), ME expresses the absolute gap (useful for effect-size interpretation), and SMD measures distributional separation in the underlying continuous score — all three are needed for a complete audit.

---
# Task 2: Intersectional Analysis (Race × Sex)

Build intersectional subgroups (race × sex), compute AIR relative to the majority reference intersection (Caucasian / Male), and report the worst-group AIR. Subgroups with n < 30 are excluded to avoid unstable estimates.

In [ ]:
# Intersectional analysis — race × sex
df['subgroup'] = df['race'] + ' / ' + df['sex']
# Keep subgroups with n >= 30
counts    = df['subgroup'].value_counts()
valid_sg  = counts[counts >= 30].index
df_sub    = df[df['subgroup'].isin(valid_sg)].copy()

sub_rates = (
    df_sub.groupby('subgroup')['high_risk']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'selection_rate', 'count': 'n'})
    .reset_index()
)

ref_rate = sub_rates.loc[
    sub_rates['subgroup'] == 'Caucasian / Male', 'selection_rate'
].values[0]

sub_rates['AIR']  = sub_rates['selection_rate'] / ref_rate
sub_rates['flag'] = sub_rates['AIR'].apply(
    lambda x: '*** BELOW 0.80' if x < 0.80 else ''
)

print(sub_rates.sort_values('AIR').to_string(index=False))

worst = sub_rates.loc[sub_rates['AIR'].idxmin()]
print(f"\nWorst: {worst['subgroup']}, AIR={worst['AIR']:.3f} and n={worst['n']}")

### Task 2 — Interpretation

The worst-performing intersectional subgroup is **Hispanic / Female**, with AIR = 0.270 and n = 82. This means Hispanic female defendants are flagged as high-risk at only 27% of the rate of Caucasian male defendants — a disparity far exceeding the EEOC 80% threshold and the most extreme AIR in the dataset.

Critically, this finding is invisible in the race-only or sex-only aggregate analyses. Examining race alone shows Hispanics close to parity (AIR ≈ 0.84 vs. Caucasian), and examining sex alone would show females receiving fewer high-risk flags than males — neither analysis reveals the extreme under-flagging concentrated on Hispanic women.

This illustrates Crenshaw's (1989) intersectionality principle operationally: discrimination can manifest at the intersection of race and sex in ways that aggregate metrics obscure. From a governance standpoint, this subgroup-level finding must be documented and assessed for the presence of business necessity, since the AIR of 0.270 easily clears the prima facie bar under any accepted disparate impact standard.

**Caution:** n = 82 is above the minimum threshold of 30 but still small. Confidence intervals should accompany this point estimate before drawing firm regulatory conclusions.

---
# Task 3: FPR and FNR Disparities by Race + Statistical Significance

- **FPR** (False Positive Rate) = P(predicted high-risk | actually did *not* recidivate)  
- **FNR** (False Negative Rate) = P(predicted low-risk  | actually *did* recidivate)  

Statistical significance is tested using a two-proportion z-test comparing African-Americans to Caucasians separately for FPR and FNR.

In [ ]:
# FPR and FNR by race
def error_rates(df, group_col, pred_col, outcome_col):
    results = []
    for grp, g in df.groupby(group_col):
        tp = ((g[pred_col]==1) & (g[outcome_col]==1)).sum()
        tn = ((g[pred_col]==0) & (g[outcome_col]==0)).sum()
        fp = ((g[pred_col]==1) & (g[outcome_col]==0)).sum()
        fn = ((g[pred_col]==0) & (g[outcome_col]==1)).sum()
        results.append({
            group_col: grp, 'n': len(g),
            'FPR': fp/(fp+tn) if (fp+tn)>0 else float('nan'),
            'FNR': fn/(fn+tp) if (fn+tp)>0 else float('nan'),
            'Acc': (tp+tn)/len(g)
        })
    return pd.DataFrame(results)

er = error_rates(df, 'race', 'high_risk', 'two_year_recid')
print(er.sort_values('FPR', ascending=False).to_string(index=False))

print()
# Highlight Black vs. White disparity
for grp in ['African-American', 'Caucasian']:
    row = er.loc[er['race'] == grp]
    print(f"{grp}: FPR={row['FPR'].values[0]:.3f}  FNR={row['FNR'].values[0]:.3f}")

In [ ]:
# ── Two-proportion z-test: FPR (African-American vs. Caucasian) ───────────
# Numerator  = non-recidivists incorrectly flagged high-risk (FP)
# Denominator = all non-recidivists in that group (FP + TN)

groups  = ['African-American', 'Caucasian']
df_fpr  = df[df['race'].isin(groups) & (df['two_year_recid'] == 0)].copy()

n_fpr   = df_fpr.groupby('race')['high_risk'].count()
ev_fpr  = df_fpr.groupby('race')['high_risk'].sum()   # high_risk=1 among non-recidivists = FP

stat_fpr, pval_fpr = proportions_ztest(
    count = ev_fpr[groups].values,
    nobs  = n_fpr[groups].values
)

print("Two-proportion z-test: FPR (African-American vs. Caucasian)")
print(f"  African-American non-recidivists flagged: {ev_fpr['African-American']:,} / {n_fpr['African-American']:,}"
      f"  = {ev_fpr['African-American']/n_fpr['African-American']:.4f}")
print(f"  Caucasian non-recidivists flagged:        {ev_fpr['Caucasian']:,} / {n_fpr['Caucasian']:,}"
      f"  = {ev_fpr['Caucasian']/n_fpr['Caucasian']:.4f}")
print(f"  z = {stat_fpr:.4f}")
print(f"  p = {pval_fpr:.6f}")
print(f"  Statistically significant (α = 0.05): {pval_fpr < 0.05}")

In [ ]:
# ── Two-proportion z-test: FNR (African-American vs. Caucasian) ───────────
# Numerator  = recidivists incorrectly predicted low-risk (FN)
# Denominator = all recidivists in that group (FN + TP)

df_fnr = df[df['race'].isin(groups) & (df['two_year_recid'] == 1)].copy()
df_fnr['missed'] = (df_fnr['high_risk'] == 0).astype(int)

n_fnr  = df_fnr.groupby('race')['missed'].count()
ev_fnr = df_fnr.groupby('race')['missed'].sum()

stat_fnr, pval_fnr = proportions_ztest(
    count = ev_fnr[groups].values,
    nobs  = n_fnr[groups].values
)

print("Two-proportion z-test: FNR (African-American vs. Caucasian)")
print(f"  African-American recidivists missed: {ev_fnr['African-American']:,} / {n_fnr['African-American']:,}"
      f"  = {ev_fnr['African-American']/n_fnr['African-American']:.4f}")
print(f"  Caucasian recidivists missed:        {ev_fnr['Caucasian']:,} / {n_fnr['Caucasian']:,}"
      f"  = {ev_fnr['Caucasian']/n_fnr['Caucasian']:.4f}")
print(f"  z = {stat_fnr:.4f}")
print(f"  p = {pval_fnr:.6f}")
print(f"  Statistically significant (α = 0.05): {pval_fnr < 0.05}")

### Task 3 — Interpretation

**FPR disparity (African-American = 0.423, Caucasian = 0.220):** Among defendants who did *not* go on to reoffend, African-Americans were incorrectly labeled high-risk at more than twice the rate of Caucasians (gap ≈ +0.203). The two-proportion z-test is statistically significant (p < 0.001), confirming this disparity is not attributable to sampling noise. In legal terms this is a disproportionate false accusation burden on an innocent Black defendant.

**FNR disparity (African-American = 0.285, Caucasian = 0.496):** Among defendants who *did* reoffend, Caucasians were missed (predicted low-risk) at nearly twice the rate of African-Americans (gap ≈ −0.211). The z-test is again statistically significant. This means the model is relatively lenient toward Caucasian recidivists and relatively severe toward African-American non-recidivists.

**Impossibility Theorem connection:** These two patterns are not independent. As shown in Lecture 03, when base recidivism rates differ between groups (which they do here — a structural consequence of historical over-policing, not individual propensity), any classifier that is calibrated across groups *cannot* simultaneously equalize FPR and FNR. COMPAS chose approximate calibration as its primary defense; the consequence is exactly the asymmetric error distribution documented above.

---
# Task 4: Publication-Quality Figure — FPR and FNR by Race

Grouped bar chart with Caucasian as the reference group. Dashed reference lines mark the Caucasian FPR and FNR.

In [ ]:
# ── Order races: Caucasian first (reference), then alphabetical ───────────
cau_fpr = er.loc[er['race'] == 'Caucasian', 'FPR'].values[0]
cau_fnr = er.loc[er['race'] == 'Caucasian', 'FNR'].values[0]

race_order = ['Caucasian'] + sorted([r for r in er['race'] if r != 'Caucasian'])
er_plot    = er.set_index('race').loc[race_order].reset_index()

x     = np.arange(len(race_order))
w     = 0.35
c_fpr = '#2166AC'   # blue
c_fnr = '#D6604D'   # red-orange

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#FAFAFA')

bars_fpr = ax.bar(x - w/2, er_plot['FPR'], w,
                  label='False Positive Rate (FPR)',
                  color=c_fpr, alpha=0.85, edgecolor='white', linewidth=0.8)
bars_fnr = ax.bar(x + w/2, er_plot['FNR'], w,
                  label='False Negative Rate (FNR)',
                  color=c_fnr, alpha=0.85, edgecolor='white', linewidth=0.8)

# Caucasian reference lines
ax.axhline(cau_fpr, color=c_fpr, linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Caucasian FPR = {cau_fpr:.3f}')
ax.axhline(cau_fnr, color=c_fnr, linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Caucasian FNR = {cau_fnr:.3f}')

# Annotate bars
for bar in bars_fpr:
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.006,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7.5, color=c_fpr)
for bar in bars_fnr:
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.006,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7.5, color=c_fnr)

# Axes and formatting
tick_labels = [
    'African-\nAmerican' if r == 'African-American' else
    'Native\nAmerican'   if r == 'Native American'  else r
    for r in race_order
]
ax.set_xticks(x)
ax.set_xticklabels(tick_labels, fontsize=10)
ax.set_xlabel('Race', fontsize=12, labelpad=8)
ax.set_ylabel('Error Rate', fontsize=12, labelpad=8)
ax.set_title(
    'False Positive and False Negative Rates by Race\n'
    '(COMPAS Risk Score, Broward County FL | Reference group: Caucasian)',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_ylim(0, max(er_plot['FPR'].max(), er_plot['FNR'].max()) * 1.20)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle=':', alpha=0.5)
ax.legend(fontsize=9, loc='upper right', framealpha=0.9)

fig.text(0.5, -0.01,
         'Source: ProPublica COMPAS analysis (n ≈ 6,172). '
         'Outcome: two_year_recid. '
         'Prediction: high_risk (score_text ≠ Low). '
         'Note: Native American n = 11 (estimate unreliable).',
         ha='center', fontsize=8, color='grey')

plt.tight_layout()
plt.savefig('fpr_fnr_by_race.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved → fpr_fnr_by_race.png")

---
# Task 5: Compliance Memo (≈ 300 words)

---

**MEMORANDUM**

**To:** Office of Algorithmic Accountability, Hypothetical Regulatory Authority  
**From:** Surafel Debebe, Internal Audit Team  
**Re:** Disparate Impact Audit — COMPAS Recidivism Risk Score (Broward County, FL)  
**Date:** April 6, 2026  
**Classification:** For Regulatory Review

---

**Purpose.** This memorandum summarizes the findings of a disparate impact audit conducted on the COMPAS recidivism risk-scoring model using the ProPublica Broward County dataset (n ≈ 6,172 post-filtering). The audit applied the burden-shifting framework established in *Griggs v. Duke Power Co.* (1971) and N.J. DCR Guidance on Algorithmic Discrimination (2025).

**Metrics Applied.** Three metrics were used in combination. The Adverse Impact Ratio (AIR) operationalizes the EEOC 80% rule, flagging any group flagged as high-risk at less than 80% or more than 125% of the Caucasian reference rate. The Marginal Effect (ME) expresses absolute selection-rate gaps. Cohen's d (SMD) measures distributional separation in the underlying continuous decile score. False Positive Rate (FPR) and False Negative Rate (FNR) capture directional error asymmetries relevant to due process and community safety considerations.

**Key Findings.** African-American defendants are flagged as high-risk at 1.74× the rate of Caucasian defendants (ME ≈ +0.245; SMD = 0.608, "large" effect). Their FPR is 0.423 versus 0.220 for Caucasians — a statistically significant gap (p < 0.001) meaning that innocent African-American defendants are incorrectly labeled high-risk at twice the rate of innocent Caucasian defendants. Caucasian recidivists are correspondingly *missed* at a higher FNR (0.496 vs. 0.285; p < 0.001). Intersectional analysis identifies Hispanic / Female defendants as the most severely under-flagged subgroup (AIR = 0.270 vs. Caucasian / Male), a finding invisible in race-only or sex-only aggregate metrics.

**Limitations.** First, the data reflect 2013–2014 Broward County, FL conditions and may not generalize geographically or temporally. Second, the Impossibility Theorem (Chouldechova 2017) establishes that simultaneous calibration and FPR parity is mathematically infeasible when base recidivism rates differ between groups; thus some disparity is structurally unavoidable under any calibrated model. Third, Native American results (n = 11) are statistically unreliable and should be excluded from formal determinations. Fourth, the logistic regression encodes `race` as a direct feature, raising a separate disparate treatment concern independent of the disparate impact findings above.

**Recommendation.** This audit establishes a prima facie case of disparate impact on African-American defendants. The organization must determine whether business necessity justifies the observed FPR disparity, document that determination in its model risk management system, evaluate less discriminatory alternatives (e.g., threshold adjustment by group, feature substitution), and implement ongoing monitoring before any deployment renewal.